# Pair Image Middle-Frame Prediction

This notebook reads folders under `raw_data/pair_test/`. Each folder must contain exactly two image files: the left/input frame and the right/input frame. It runs EMA-VFI-small, AMT-S, and Practical-RIFE v4.25 sequentially, then writes visual triplets under `notebooks/outputs/<pair_folder>/<model_name>/`.

Each output model folder contains:

- `left.png`
- `middle.png`
- `right.png`

Use a kernel from the project `uv` environment so the local package dependencies and CUDA-enabled PyTorch are available.

In [6]:
from pathlib import Path
import gc
import sys

import numpy as np
from PIL import Image
import torch

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").is_dir():
    raise RuntimeError(f"Could not locate project root from {Path.cwd()}")

src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from video_interpolation.adapters.ema_vfi import EMAVFIAdapter, EMAVFIAdapterConfig
from video_interpolation.adapters.amt import AMTAdapter, AMTAdapterConfig
from video_interpolation.adapters.rife import PracticalRIFEAdapter, PracticalRIFEAdapterConfig
from video_interpolation.image_io import tensor_to_uint8_hwc, uint8_hwc_to_tensor

torch.set_grad_enabled(False)

INPUT_ROOT = PROJECT_ROOT / "raw_data" / "pair_test"
OUTPUT_ROOT = PROJECT_ROOT / "notebooks" / "outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}

print(f"Project root: {PROJECT_ROOT}")
print(f"Input root:   {INPUT_ROOT}")
print(f"Output root:  {OUTPUT_ROOT}")
print(f"CUDA:         {torch.cuda.is_available()}")

Project root: /home/lighter_01/projects/itmo/ai_architecture/video_interpolation
Input root:   /home/lighter_01/projects/itmo/ai_architecture/video_interpolation/raw_data/pair_test
Output root:  /home/lighter_01/projects/itmo/ai_architecture/video_interpolation/notebooks/outputs
CUDA:         True


In [7]:
def pair_directories(input_root: Path) -> list[Path]:
    if not input_root.is_dir():
        raise FileNotFoundError(f"Input directory does not exist: {input_root}")
    return sorted(path for path in input_root.iterdir() if path.is_dir())


def pair_images(pair_dir: Path) -> tuple[Path, Path]:
    images = sorted(
        path for path in pair_dir.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )
    if len(images) != 2:
        raise ValueError(f"Expected exactly two images in {pair_dir}, found {len(images)}: {images}")
    return images[0], images[1]


def load_rgb_tensor(path: Path) -> torch.Tensor:
    image = Image.open(path).convert("RGB")
    return uint8_hwc_to_tensor(np.asarray(image))


def save_rgb_copy(source_path: Path, output_path: Path) -> None:
    Image.open(source_path).convert("RGB").save(output_path)


def save_prediction_tensor(tensor: torch.Tensor, output_path: Path) -> None:
    Image.fromarray(tensor_to_uint8_hwc(tensor)).save(output_path)


def release_gpu_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


pairs = pair_directories(INPUT_ROOT)
if not pairs:
    raise RuntimeError(f"No pair folders found under {INPUT_ROOT}")

for pair_dir in pairs:
    left_path, right_path = pair_images(pair_dir)
    print(f"{pair_dir.name}: {left_path.name} -> {right_path.name}")

001: frame1.png -> frame2.png
002: frame1.png -> frame2.png
003: frame1.png -> frame2.png


In [8]:
for name in list(sys.modules):
    if name == "model" or name.startswith("model.") or name == "train_log" or name.startswith("train_log."):
        sys.modules.pop(name, None)

In [11]:
MODEL_SPECS = [
    (
        "ema_vfi_small",
        EMAVFIAdapter,
        EMAVFIAdapterConfig(),
    ),
    (
        "practical_rife_v4_25",
        PracticalRIFEAdapter,
        PracticalRIFEAdapterConfig(),
    ),
    (
        "amt_s",
        AMTAdapter,
        AMTAdapterConfig(),
    ),
]

In [12]:
results = []

for model_name, adapter_cls, config in MODEL_SPECS:
    print(f"\nLoading {model_name}...")
    adapter = adapter_cls(config)
    try:
        adapter.load_checkpoint()
        adapter.c
        print(f"Loaded {model_name}")

        for pair_dir in pairs:
            left_path, right_path = pair_images(pair_dir)
            left = load_rgb_tensor(left_path)
            right = load_rgb_tensor(right_path)
            middle = adapter.predict_pair(left, right)

            output_dir = OUTPUT_ROOT / pair_dir.name / model_name
            output_dir.mkdir(parents=True, exist_ok=True)
            save_rgb_copy(left_path, output_dir / "left.png")
            save_prediction_tensor(middle, output_dir / "middle.png")
            save_rgb_copy(right_path, output_dir / "right.png")

            results.append(
                {
                    "pair": pair_dir.name,
                    "model": model_name,
                    "output_dir": output_dir,
                }
            )
            print(f"  wrote {output_dir.relative_to(PROJECT_ROOT)}")
    finally:
        adapter.close()
        del adapter
        release_gpu_memory()

print(f"\nDone. Wrote {len(results)} prediction folders under {OUTPUT_ROOT.relative_to(PROJECT_ROOT)}")


Loading ema_vfi_small...


RuntimeError: Error(s) in loading state_dict for MultiScaleFlow:
	Unexpected key(s) in state_dict: "feature_bone.block4.2.norm1.weight", "feature_bone.block4.2.norm1.bias", "feature_bone.block4.2.attn.q.weight", "feature_bone.block4.2.attn.q.bias", "feature_bone.block4.2.attn.kv.weight", "feature_bone.block4.2.attn.kv.bias", "feature_bone.block4.2.attn.cor_embed.weight", "feature_bone.block4.2.attn.cor_embed.bias", "feature_bone.block4.2.attn.proj.weight", "feature_bone.block4.2.attn.proj.bias", "feature_bone.block4.2.attn.motion_proj.weight", "feature_bone.block4.2.attn.motion_proj.bias", "feature_bone.block4.2.norm2.weight", "feature_bone.block4.2.norm2.bias", "feature_bone.block4.2.mlp.fc1.weight", "feature_bone.block4.2.mlp.fc1.bias", "feature_bone.block4.2.mlp.dwconv.dwconv.weight", "feature_bone.block4.2.mlp.dwconv.dwconv.bias", "feature_bone.block4.2.mlp.fc2.weight", "feature_bone.block4.2.mlp.fc2.bias", "feature_bone.block4.3.norm1.weight", "feature_bone.block4.3.norm1.bias", "

In [ ]:
for row in results:
    print(f"{row['pair']} / {row['model']}: {row['output_dir'].relative_to(PROJECT_ROOT)}")